In [1]:
!pip install requests pandas transformers torch

In [2]:
import requests
import pandas as pd
from transformers import pipeline

In [3]:
WHALE_ALERT_NEWS="https://whale-alert.io/news.json?range=last_30_days&visibility=1,0,0"

In [4]:
WHALE_ALERT_NEWS1="https://whale-alert.io/news.json?range=last_30_days&visibility=1,1,0"

In [5]:
response_w = requests.get(WHALE_ALERT_NEWS)
data_w = response_w.json()

In [6]:
response_w1 = requests.get(WHALE_ALERT_NEWS1)
data_w1 = response_w1.json()

In [7]:
data_w[0]

{'id': 'b66c92fc7d58',
 'headline': 'Prysm consensus-client bug after Fusaka upgrade caused validator outages, missed epochs and ~382\u202fETH in lost rewards; fixes deployed and network recovered',
 'description': "A Prysm client bug that appeared immediately after Ethereum's Fusaka upgrade (epoch 411392, Dec 4, 2025, 21:49 UTC) triggered expensive state recomputation from certain attestations, exhausting resources and degrading Prysm validator performance. Validator participation for Prysm fell from >95% to ~75%, the network missed 41 epochs and about 382 ETH in proof rewards were lost. Prysm represented ~15%–22.71% of validators. Emergency runtime flags were used and permanent fixes shipped in v7.0.1 and v7.1.0. Client diversity (other clients kept ~75%–85% of validators) prevented finality loss and the network returned to ~99% participation within 24 hours. The Fusaka upgrade itself succeeded and added PeerDAS to increase blob capacity for Layer‑2 scaling.",
 'emoticons': '⚠',
 'ic

In [8]:
data_w1[0]

{'id': 'b76c06f28325',
 'headline': "Cardano (ADA) argued to have more upside than Bitcoin (BTC) as ETF-driven demand limits BTC's near-term gains",
 'description': "Comparison of Bitcoin and Cardano: Bitcoin is described as a dominant $1.8 trillion monetary crypto with roughly 95% of its supply mined and saw major demand growth after spot Bitcoin ETFs launched in January 2024, which the article says limits Bitcoin's near-term upside (though a 3x–5x gain over a decade is plausible). Cardano (ADA) is presented as having one of its largest growth opportunities ahead and therefore more upside potential for investors seeking a smaller, higher-growth asset. The piece also notes gold rose about 53% over the past 12 months while Bitcoin was flat.",
 'emoticons': '⚖️',
 'icon': 'news-icons/btc.png',
 'tags': ['*Bitcoin', 'Altcoin', 'Etf', 'Price', 'Trading'],
 'articles': [{'headline': 'Which Cryptocurrency Has More Upside? Bitcoin vs. Cardano',
   'url': 'https://finance.yahoo.com/news/crypto

In [9]:
len(data_w)

50

In [10]:
len(data_w1)

88

In [11]:
list_w=[]

In [12]:
for item in data_w:
    list_w.append({
        "title": item["headline"],
        "description":item["description"],
        "published_at": item["published_at"],
        "tags": item["tags"]
    })
for item in data_w1:
    list_w.append({
        "title": item["headline"],
        "description":item["description"],
        "published_at": item["published_at"],
        "tags": item["tags"]
    })

df = pd.DataFrame(list_w)

In [13]:
df.head()

,title,description,published_at,tags
0,Prysm consensus-client bug after Fusaka upgrad...,A Prysm client bug that appeared immediately a...,1765737000,"[Ethereum, Consensus, Staking, Security, Funda..."
1,Crypto winter sparks sell-off in public crypto...,"After Bitcoin's October crash, more than 180 p...",1765724410,"[Bitcoin, Ethereum, Trading, Price]"
2,Bank of Japan 25 bps hike (98% priced) could t...,Bitcoin is under heavy selling pressure ahead ...,1765713000,"[Bitcoin, Trading, Government, Price, Predicti..."
3,Coinbase XRP open interest rises while price s...,Open interest for XRP on Coinbase has increase...,1765712040,"[Trading, Altcoin, Price]"
4,Cardano-linked privacy token Midnight (NIGHT) ...,"Midnight (NIGHT), a Cardano‑linked privacy tok...",1765704660,"[Altcoin, Privacy, Trading, Layer 1]"


In [14]:
df["text"] = df["title"] + ". " + df["description"].str.strip()
df = df[df["text"] != ""]

In [15]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cpu


In [16]:
texts = df["text"].astype(str).tolist()
results = sentiment_pipeline(
    texts,
    truncation=True
)

df["sentiment"] = [r["label"].lower() for r in results]
df["confidence"] = [r["score"] for r in results]


In [17]:
df.head()

,title,description,published_at,tags,text,sentiment,confidence
0,Prysm consensus-client bug after Fusaka upgrad...,A Prysm client bug that appeared immediately a...,1765737000,"[Ethereum, Consensus, Staking, Security, Funda...",Prysm consensus-client bug after Fusaka upgrad...,negative,0.955347
1,Crypto winter sparks sell-off in public crypto...,"After Bitcoin's October crash, more than 180 p...",1765724410,"[Bitcoin, Ethereum, Trading, Price]",Crypto winter sparks sell-off in public crypto...,negative,0.974643
2,Bank of Japan 25 bps hike (98% priced) could t...,Bitcoin is under heavy selling pressure ahead ...,1765713000,"[Bitcoin, Trading, Government, Price, Predicti...",Bank of Japan 25 bps hike (98% priced) could t...,negative,0.959047
3,Coinbase XRP open interest rises while price s...,Open interest for XRP on Coinbase has increase...,1765712040,"[Trading, Altcoin, Price]",Coinbase XRP open interest rises while price s...,negative,0.901479
4,Cardano-linked privacy token Midnight (NIGHT) ...,"Midnight (NIGHT), a Cardano‑linked privacy tok...",1765704660,"[Altcoin, Privacy, Trading, Layer 1]",Cardano-linked privacy token Midnight (NIGHT) ...,neutral,0.795211


In [18]:
df.to_csv("crypto_news_finbert_sentiment_whale_news.csv", index=False)
print("Saved results to crypto_news_finbert_sentiment.csv")

Saved results to crypto_news_finbert_sentiment.csv


In [19]:
print(df[["title","description","published_at","text","sentiment","confidence"]].head())

                                               title  \
0  Prysm consensus-client bug after Fusaka upgrad...   
1  Crypto winter sparks sell-off in public crypto...   
2  Bank of Japan 25 bps hike (98% priced) could t...   
3  Coinbase XRP open interest rises while price s...   
4  Cardano-linked privacy token Midnight (NIGHT) ...   

                                         description  published_at  \
0  A Prysm client bug that appeared immediately a...    1765737000   
1  After Bitcoin's October crash, more than 180 p...    1765724410   
2  Bitcoin is under heavy selling pressure ahead ...    1765713000   
3  Open interest for XRP on Coinbase has increase...    1765712040   
4  Midnight (NIGHT), a Cardano‑linked privacy tok...    1765704660   

                                                text sentiment  confidence  
0  Prysm consensus-client bug after Fusaka upgrad...  negative    0.955347  
1  Crypto winter sparks sell-off in public crypto...  negative    0.974643  
2  Bank of 